# PBL6 — NOTEBOOK 01: HUẤN LUYỆN VÀ ĐỐI SÁNH MÔ HÌNH HỌC MÁY ĐA GIAO THỨC WEB API
## Nền Tảng An Ninh Web API Thông Minh & Tác Tử Tự Động Diễn Tập Tấn Công Trên Cyber Range Phân Tán

> **Tác giả:** Trần Văn Công (Kỹ sư An ninh AI / Tech Lead — Thành viên A)
> **Cơ sở khoa học & Tiêu chuẩn áp dụng:**
> - [Ref 08] Torrano-Gimenez et al. (Wiley SCN 2015): Không gian 17 đặc trưng hình thái, ký tự và từ khóa an ninh.
> - [Ref 09] IEEE Access 2024 / MDPI Electronics 2025: Khảo sát đối sánh thực nghiệm giữa 5 trường phái thuật toán học máy.
> - [Ref 15 & 16] Dự án Tiêu chuẩn An toàn OWASP Benchmark: Chỉ số độ nhạy thực tế Youden's Index $J = \text{TPR} - \text{FPR}$.
> 
> **Mục tiêu cốt lõi của Notebook:**
> 1. **Nạp tập dữ liệu tinh hoa đa giao thức** (`curated_deduped_dataset.csv` đã qua bộ lọc trùng Canonical SHA-256) bao phủ đầy đủ: **REST, GraphQL, SOAP/XML, JSON-RPC, Generic Web**.
> 2. Khảo sát và trực quan hóa phân bổ dữ liệu theo giao thức Web API và các lớp tấn công (SQLi, XSS, Path Traversal, CMD Injection, XXE, Anomaly Tamper).
> 3. **Trích xuất 17 chiều đặc trưng chuẩn hóa** (Canonical Feature Vector) qua pipeline siêu tốc.
> 4. **Huấn luyện và đối sánh 5 trường phái thuật toán ứng viên (Multi-Model Benchmarking)**:
>    - *Logistic Regression (Mô hình Tuyến tính Cơ sở)*
>    - *Decision Tree CART (Cây Quyết định Đơn lẻ)*
>    - *Linear SVM Calibrated (Siêu phẳng Cực đại Lề)*
>    - *Random Forest (Học kết hợp Đóng bao - Bagging Ensemble)*
>    - *XGBoost (Học kết hợp Tăng cường - Boosting Ensemble)*
> 5. Đánh giá toàn diện 6 chỉ số: Độ chính xác (Accuracy), Precision Macro, Recall Macro, F1-Macro, Youden's Index J và Độ trễ suy luận SLA ($< 2\text{ ms/req}$).
> 6. **Lựa chọn mô hình xuất sắc nhất (Champion Model)** và tinh chỉnh siêu tham số tối ưu (GridSearchCV).
> 7. Huấn luyện mô hình Phát hiện dị thường không giám sát (Isolation Forest) trên baseline Benign đa giao thức để bắt Zero-Day.
> 8. **Xuất bản Model Artifacts** (`rf_model.joblib`, `iforest_model.joblib`, metadata JSON) phục vụ tích hợp trực tiếp vào WAF Gateway.

In [ ]:
# 1. Thiết lập đường dẫn môi trường và nạp các thư viện cần thiết
import json
import sys
import time
import hashlib
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from xgboost import XGBClassifier

# Thiết lập cấu hình hiển thị đồ thị chuẩn học thuật
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

# Tự động định vị thư mục gốc dự án PBL6
repo_root = None
for p in [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve().parent.parent, Path(r"C:\Study\HocKy6\PBL6")]:
    if (p / 'data').exists() and (p / 'gateway').exists():
        repo_root = p
        break
if repo_root is None:
    repo_root = Path(r"C:\Study\HocKy6\PBL6")

ml_dir = repo_root / 'ml-engine'
gateway_dir = repo_root / 'gateway'
for p in [str(repo_root), str(ml_dir), str(gateway_dir)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from features.extractor import CANONICAL_FEATURE_NAMES, extract_batch_vectors
from app.security.protocol_detector import ApiProtocolDetector, ApiProtocol

print(f"[+] Thư mục gốc đề tài: {repo_root}")
print(f"[+] Không gian 17 đặc trưng chuẩn tắc: Đã nạp {len(CANONICAL_FEATURE_NAMES)} đặc trưng")
print("[+] Bộ nhận diện đa giao thức & thư viện học máy đã khởi tạo sẵn sàng!")

## 2. Nạp Tập Dữ Liệu Tinh Hoa Đa Giao Thức (Curated Multi-Protocol Web API Dataset)

Tập dữ liệu được tổng hợp từ nguồn quy mô lớn (> 6,3 triệu request từ OpenAppSec, CSIC 2010, ModSecurity) sau khi đã chạy qua bộ **Canonical Deduplicator (loại bỏ 18% - 44% mẫu trùng lặp)** và **gắn nhãn tự động theo giao thức Web API** (`REST`, `GRAPHQL`, `SOAP_XML`, `JSON_RPC`, `GENERIC_WEB`).

In [ ]:
# 2. Nạp và phân tích phân bổ dữ liệu
CURATED_CSV_PRIMARY = Path(r"C:\Mod\muse-data-exchange\pbl6-datasets\curated_deduped_dataset.csv")
FALLBACK_TRAIN_CSV = repo_root / 'data' / 'processed' / 'defense' / 'train.csv'
FALLBACK_VAL_CSV = repo_root / 'data' / 'processed' / 'defense' / 'val.csv'

if CURATED_CSV_PRIMARY.exists():
    print(f"[+] Đang nạp tập dữ liệu tinh hoa đa giao thức: {CURATED_CSV_PRIMARY} ...")
    df_raw = pd.read_csv(CURATED_CSV_PRIMARY)
    print(f" -> Nạp thành công {len(df_raw):,} bản ghi ({CURATED_CSV_PRIMARY.stat().st_size / (1024*1024):.2f} MB)")
    
    # Phân chia phân tầng: 80% Tập Huấn Luyện (Train), 20% Tập Kiểm Định (Validation)
    strat_col = df_raw['label'].astype(str)
    df_train, df_val = train_test_split(df_raw, test_size=0.2, random_state=42, stratify=strat_col)
    df_train = df_train.reset_index(drop=True)
    df_val = df_val.reset_index(drop=True)
else:
    print(f"[!] Không tìm thấy tập curated ngoài ổ đĩa. Tự động chuyển sang dữ liệu nội sinh Bookie Bookstore ...")
    df_train = pd.read_csv(FALLBACK_TRAIN_CSV)
    df_val = pd.read_csv(FALLBACK_VAL_CSV)

print(f"\n[+] Quy mô Tập Huấn luyện (Train Set):   {len(df_train):,} mẫu")
print(f"[+] Quy mô Tập Kiểm định (Val Set):       {len(df_val):,} mẫu")

# Phân bổ theo Giao thức Web API
if 'api_protocol' in df_train.columns:
    print("\n=== Phân Bổ Giao Thức Web API (Tập Huấn Luyện) ===")
    display(df_train['api_protocol'].value_counts().to_frame())

print("\n=== Phân Bổ Lớp Tấn Công & Lưu Lượng Sạch (Tập Huấn Luyện) ===")
display(df_train['attack_type'].value_counts().to_frame())

print("\nVí dụ 3 dòng đầu tiên:")
df_train[['method', 'path', 'label', 'attack_type', 'api_protocol']].head(3)

## 3. Trích Xuất Không Gian 17 Chiều Đặc Trưng Chuẩn Hóa (Feature Extraction)

Chuyển đổi từng request HTTP thô thành vector số học 17 chiều chuẩn hóa:
- **12 Đặc trưng hình thái & entropy:** Độ dài chuỗi, tỷ lệ ký tự đặc biệt, Shannon Entropy, số lượng số, dấu câu, thẻ HTML/SQL delimiters...
- **5 Đặc trưng cú pháp & từ khóa:** Tần suất từ khóa nguy hiểm cho SQLi, XSS, Path Traversal, Command Injection.

In [ ]:
# 3. Pipeline trích xuất 17 đặc trưng số học
CANONICAL_CLASS_MAP = {
    'BENIGN': 'BENIGN',
    'SQLI': 'SQLI',
    'XSS': 'XSS',
    'PATH': 'PATH_TRAVERSAL',
    'PATH_TRAVERSAL': 'PATH_TRAVERSAL',
    'CMD': 'COMMAND_INJECTION',
    'COMMAND_INJECTION': 'COMMAND_INJECTION',
    'XXE_INJECTION': 'COMMAND_INJECTION',
    'ANOMALY_TAMPER': 'SQLI',
    'GRAPHQL_RECURSION_DOS': 'COMMAND_INJECTION',
    'GRAPHQL_INTROSPECTION_PROBE': 'SQLI',
}
CANONICAL_CLASSES = ['BENIGN', 'SQLI', 'XSS', 'PATH_TRAVERSAL', 'COMMAND_INJECTION']

print("Đang trích xuất vector 17 đặc trưng cho Tập Huấn Luyện (Train Set)...", flush=True)
t0 = time.perf_counter()
X_train = extract_batch_vectors(df_train.to_dict('records'))
y_train = df_train['attack_type'].map(lambda x: CANONICAL_CLASS_MAP.get(str(x).upper().strip(), 'BENIGN')).values
t_train = time.perf_counter() - t0
print(f" -> Trích xuất thành công X_train: {X_train.shape} trong {t_train:.2f}s ({len(X_train)/t_train:.0f} req/s)")

print("Đang trích xuất vector 17 đặc trưng cho Tập Kiểm Định (Validation Set)...", flush=True)
t0 = time.perf_counter()
X_val = extract_batch_vectors(df_val.to_dict('records'))
y_val = df_val['attack_type'].map(lambda x: CANONICAL_CLASS_MAP.get(str(x).upper().strip(), 'BENIGN')).values
t_val = time.perf_counter() - t0
print(f" -> Trích xuất thành công X_val:   {X_val.shape} trong {t_val:.2f}s ({len(X_val)/t_val:.0f} req/s)")

## 4. Huấn Luyện & Đối Sánh 5 Trường Phái Thuật Toán (Multi-Model Benchmarking)

Thực hiện đối sánh 5 trường phái thuật toán học máy đại diện:
1. **Mô hình Tuyến tính Cơ sở:** Logistic Regression (có StandardScaler).
2. **Cây Quyết định Đơn lẻ:** Decision Tree (CART).
3. **Siêu phẳng Cực đại Lề:** Linear SVM (Calibrated predict_proba, có StandardScaler).
4. **Học kết hợp Đóng bao (Bagging):** Random Forest (RF) — 100 cây.
5. **Học kết hợp Tăng cường (Boosting):** XGBoost (GBDT).

Các chỉ số đánh giá:
- **Accuracy / Precision / Recall / F1-Macro**.
- **Youden's Index $J = \text{TPR} - \text{FPR}$** (Đo lường độ nhạy thực tế, cân đối giữa bắt độc và chặn nhầm).
- **Độ trễ suy luận (ms/req)** (Đo lường thời gian suy luận trên CPU, yêu cầu đạt chuẩn WAF Gateway SLA < 2.0 ms).

In [ ]:
# 4. Khởi tạo danh sách mô hình ứng viên và thực hiện benchmark
label_to_idx = {name: i for i, name in enumerate(CANONICAL_CLASSES)}
idx_to_label = {i: name for i, name in enumerate(CANONICAL_CLASSES)}
y_train_num = np.array([label_to_idx[c] for c in y_train])
y_val_num = np.array([label_to_idx[c] for c in y_val])

candidate_models = [
    ("Logistic Regression (Scaled)", "Mô hình Tuyến tính", make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, random_state=42)), False),
    ("Decision Tree (CART)", "Cây Đơn lẻ", DecisionTreeClassifier(max_depth=12, random_state=42), False),
    ("Linear SVM (Scaled)", "Siêu phẳng Cực đại Lề", make_pipeline(StandardScaler(), CalibratedClassifierCV(LinearSVC(dual=False, max_iter=500, random_state=42))), False),
    ("Random Forest", "Bagging Ensemble", RandomForestClassifier(n_estimators=100, max_depth=16, min_samples_split=4, class_weight='balanced', random_state=42, n_jobs=-1), False),
    ("XGBoost", "Boosting Ensemble", XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, eval_metric='mlogloss', random_state=42, n_jobs=-1), True),
]

benchmark_results = []
fitted_models = {}

print(f"{'Tên Mô Hình':<28} | {'Trường Phái':<22} | {'F1-Macro':<8} | {'Accuracy':<8} | {'Độ Trễ':<10}")
print("-" * 86)

for i, (model_name, paradigm, model_obj, is_numeric_label) in enumerate(candidate_models):
    print(f"[{i+1}/5] Đang huấn luyện {model_name} ...", end=" ", flush=True)
    y_tr = y_train_num if is_numeric_label else y_train
    y_vl = y_val_num if is_numeric_label else y_val
    
    # Đo thời gian huấn luyện
    t_start = time.perf_counter()
    model_obj.fit(X_train, y_tr)
    train_time = time.perf_counter() - t_start
    fitted_models[model_name] = model_obj
    
    # Đo độ trễ suy luận trên tập Validation
    t_lat0 = time.perf_counter()
    y_pred_raw = model_obj.predict(X_val)
    t_lat = (time.perf_counter() - t_lat0) * 1000.0 / len(X_val)
    
    # Chuyển đổi nhãn dự đoán
    if is_numeric_label:
        y_pred = np.array([idx_to_label[idx] for idx in y_pred_raw])
    else:
        y_pred = y_pred_raw
        
    # Tính toán các chỉ số đánh giá
    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_val, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_val, y_pred, average='macro', zero_division=0)
    
    # Tính chỉ số Youden's Index J = TPR - FPR
    is_attack_true = (y_val != 'BENIGN')
    is_attack_pred = (y_pred != 'BENIGN')
    tpr = np.sum(is_attack_true & is_attack_pred) / np.sum(is_attack_true)
    fpr = np.sum((~is_attack_true) & is_attack_pred) / np.sum(~is_attack_true)
    youden_j = tpr - fpr
    
    benchmark_results.append({
        'model_name': model_name,
        'paradigm': paradigm,
        'accuracy': round(acc * 100, 2),
        'precision_macro': round(prec * 100, 2),
        'recall_macro': round(rec * 100, 2),
        'f1_macro': round(f1 * 100, 2),
        'youden_index_j': round(youden_j, 4),
        'avg_latency_ms': round(t_lat, 4),
        'train_time_sec': round(train_time, 2)
    })
    
    print(f"xong trong {train_time:.2f}s | F1: {f1*100:6.2f}% | Độ trễ: {t_lat:6.4f} ms")

df_results = pd.DataFrame(benchmark_results)
print("\nBẢNG ĐỐI SÁNH HIỆU NĂNG 5 MÔ HÌNH:")
display(df_results)

## 5. Trực Quan Hóa & So Sánh Hiệu Năng Các Mô Hình Ứng Viên

Vẽ biểu đồ đối sánh trực quan 4 chỉ số cốt lõi: F1-Macro, Accuracy, Youden's Index J và Độ trễ suy luận.

In [ ]:
# 5. Trực quan hóa đối sánh hiệu năng các mô hình
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. F1-Score Macro
sns.barplot(data=df_results, x='model_name', y='f1_macro', ax=axes[0, 0], palette='viridis', hue='model_name', legend=False)
axes[0, 0].set_title('Chỉ số F1-Score (Macro) giữa 5 Mô Hình (%)', fontsize=12, fontweight='bold')
axes[0, 0].set_ylim(50, 102)
axes[0, 0].set_ylabel('F1-Macro (%)')
axes[0, 0].tick_params(axis='x', rotation=20)
for p in axes[0, 0].patches:
    axes[0, 0].annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=9)

# 2. Youden's Index J
sns.barplot(data=df_results, x='model_name', y='youden_index_j', ax=axes[0, 1], palette='crest', hue='model_name', legend=False)
axes[0, 1].set_title("Chỉ số Youden's Index J (Độ nhạy an ninh: TPR - FPR)", fontsize=12, fontweight='bold')
axes[0, 1].set_ylim(0.5, 1.05)
axes[0, 1].set_ylabel('Chỉ số Youden J')
axes[0, 1].tick_params(axis='x', rotation=20)
for p in axes[0, 1].patches:
    axes[0, 1].annotate(f"{p.get_height():.4f}", (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=9)

# 3. Độ trễ suy luận so với chuẩn SLA
sns.barplot(data=df_results, x='model_name', y='avg_latency_ms', ax=axes[1, 0], palette='magma', hue='model_name', legend=False)
axes[1, 0].axhline(y=2.0, color='red', linestyle='--', linewidth=1.5, label='Ngưỡng SLA Gateway (< 2.0 ms)')
axes[1, 0].set_title('Độ Trễ Suy Luận Trung Bình Mỗi Request (ms/req)', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Độ trễ (ms)')
axes[1, 0].tick_params(axis='x', rotation=20)
axes[1, 0].legend()
for p in axes[1, 0].patches:
    axes[1, 0].annotate(f"{p.get_height():.4f} ms", (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=9)

# 4. Thời gian huấn luyện
sns.barplot(data=df_results, x='model_name', y='train_time_sec', ax=axes[1, 1], palette='rocket', hue='model_name', legend=False)
axes[1, 1].set_title('Thời Gian Huấn Luyện Mô Hình (giây)', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Thời gian (giây)')
axes[1, 1].tick_params(axis='x', rotation=20)
for p in axes[1, 1].patches:
    axes[1, 1].annotate(f"{p.get_height():.2f}s", (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Lựa Chọn Champion Model & Tinh Chỉnh Siêu Tham Số (Hyperparameter Tuning)

**Random Forest Classifier** đạt sự cân bằng tối ưu: F1-Score vượt trội ($> 95\%$), độ bền chống Overfitting cao, và suy luận đa luồng $< 0.1\text{ ms/req}$, hoàn toàn thỏa mãn SLA khắt khe của WAF Gateway. Tiến hành tối ưu tham số với GridSearchCV.

In [ ]:
# 6. Tinh chỉnh siêu tham số Champion Random Forest
print("[+] Đang tinh chỉnh siêu tham số cho Champion Random Forest bằng GridSearchCV ...", flush=True)
param_grid = {
    'max_depth': [14, 18],
    'min_samples_split': [2, 4],
}

base_rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
grid_search = GridSearchCV(base_rf, param_grid, cv=2, scoring='f1_macro', n_jobs=1, verbose=1)

t0 = time.perf_counter()
grid_search.fit(X_train, y_train)
tuning_time = time.perf_counter() - t0

champion_rf = grid_search.best_estimator_
best_params = grid_search.best_params_

print(f"\n[+] Tinh chỉnh GridSearchCV hoàn tất trong {tuning_time:.2f} giây")
print(f"    - Bộ siêu tham số tối ưu:        {best_params}")
print(f"    - Điểm F1-Macro trung bình (CV):  {grid_search.best_score_ * 100:.2f}%")

# Đánh giá lại trên Tập Kiểm định (Validation Set)
val_pred = champion_rf.predict(X_val)
val_acc = accuracy_score(y_val, val_pred)
val_f1 = f1_score(y_val, val_pred, average='macro', zero_division=0)
print(f"    - F1-Score trên Tập Kiểm định:    {val_f1 * 100:.2f}%")
print(f"    - Độ chính xác trên Tập Kiểm định: {val_acc * 100:.2f}%")

## 7. Huấn Luyện Mô Hình Phát Hiện Bất Thường (Isolation Forest — Anomaly Detector)

Mô hình học máy không giám sát (Unsupervised) được huấn luyện **hoàn toàn trên dữ liệu Benign đa giao thức** để xây dựng baseline chuẩn tắc cho hành vi bình thường. Bất kỳ mẫu lưu lượng nào có phân phối đặc trưng lệch xa baseline sẽ bị chấm điểm dị biệt (Anomaly Score) cao để bắt các cuộc tấn công Zero-Day.

In [ ]:
# 7. Huấn luyện mô hình Isolation Forest trên lưu lượng sạch baseline
print("[+] Đang lọc các vector lưu lượng sạch (Benign) làm baseline cho Isolation Forest ...", flush=True)
benign_mask = (y_train == 'BENIGN')
X_train_benign = X_train[benign_mask]

print(f" -> Huấn luyện Isolation Forest trên {len(X_train_benign):,} mẫu Benign đa giao thức ...", flush=True)
iforest_model = IsolationForest(
    n_estimators=100,
    contamination=0.03,
    max_samples=min(5000, len(X_train_benign)),
    random_state=42,
    n_jobs=-1,
)
t0 = time.perf_counter()
iforest_model.fit(X_train_benign)
print(f" -> Huấn luyện Isolation Forest thành công trong {time.perf_counter() - t0:.2f} giây!")

## 8. Xuất Bản Model Artifacts & Đồng Bộ Chữ Ký Số SHA-256

Lưu trữ các mô hình đã huấn luyện vào `ml-engine/artifacts/` và tự động cập nhật metadata có kèm mã băm mật mã học SHA-256 để chống can thiệp trái phép (Model Poisoning).

In [ ]:
# 8. Xuất bản Model Artifacts & Đồng bộ mã băm SHA-256
artifacts_dir = repo_root / 'ml-engine' / 'artifacts'
artifacts_dir.mkdir(parents=True, exist_ok=True)

# 1. Lưu mô hình Champion Random Forest
rf_path = artifacts_dir / 'rf_model.joblib'
joblib.dump(champion_rf, rf_path, compress=3)
with open(rf_path, 'rb') as f:
    rf_hash = hashlib.sha256(f.read()).hexdigest()

# 2. Lưu mô hình Isolation Forest
if_path = artifacts_dir / 'iforest_model.joblib'
joblib.dump(iforest_model, if_path, compress=3)
with open(if_path, 'rb') as f:
    if_hash = hashlib.sha256(f.read()).hexdigest()

# 3. Lưu bảng tổng kết benchmark JSON
bench_path = artifacts_dir / 'benchmark_summary.json'
with open(bench_path, 'w', encoding='utf-8') as f:
    json.dump(benchmark_results, f, indent=2, ensure_ascii=False)

# 4. Lưu Metadata & mã băm SHA-256 của Random Forest
rf_metadata = {
    'model_name': 'RandomForestClassifier',
    'champion_role': 'Primary Supervised Classifier',
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'sha256_hash': rf_hash,
    'best_hyperparameters': best_params,
    'classes': CANONICAL_CLASSES,
    'feature_names': CANONICAL_FEATURE_NAMES,
    'validation_metrics': {
        'accuracy': round(val_acc * 100, 2),
        'f1_macro': round(val_f1 * 100, 2)
    }
}
with open(artifacts_dir / 'rf_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(rf_metadata, f, indent=2, ensure_ascii=False)

# 5. Lưu Metadata & mã băm SHA-256 của Isolation Forest
if_metadata = {
    'model_name': 'IsolationForest',
    'role': 'Unsupervised Anomaly Detector',
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'sha256_hash': if_hash,
    'feature_names': CANONICAL_FEATURE_NAMES,
    'contamination': 0.03,
}
with open(artifacts_dir / 'iforest_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(if_metadata, f, indent=2, ensure_ascii=False)

print("[+] ĐÃ XUẤT BẢN VÀ ĐỒNG BỘ CHỮ KÝ SHA-256 CHO TOÀN BỘ ARTIFACTS THÀNH CÔNG!")
print(f"    - Champion RF Model: {rf_path} (SHA-256: {rf_hash[:16]}...)")
print(f"    - Isolation Forest:  {if_path} (SHA-256: {if_hash[:16]}...)")
print(f"    - RF Metadata:       {artifacts_dir / 'rf_metadata.json'}")
print(f"    - IForest Metadata:  {artifacts_dir / 'iforest_metadata.json'}")